<br>

# Phase 3 — grounding & the converging-evidence decision

---

This notebook is the heart of zlabel. One `Labeler.label()` call runs the full six-step annotation
loop from `docs/design.md` (this notebook focuses on grounding + the decision; the separate Phase 4a
later added the namer and guardrail in steps 3–4):

1. Normalize symbols *(Phase 2)*
2. Score against panels — a coarse prior *(Phase 2)*
3. Converge on ZFA anatomy — the namer *(Phase 4a)*
4. Guardrail against the panel's ontology anchor *(Phase 4a)*
5. **Decide** — assign with confidence, roll up to a coarser tier, or abstain honestly *(Phase 3)*
6. **Emit** a `Label` evidence packet *(Phase 3)*

Phase 2 produced a ranked bucket table. Phase 3 turns that table into a decision: one
`Labeler(stage_hpf=…).label([...])` call runs the whole loop and returns a `Label` — or an honest
`mixed/unresolved` when the evidence does not converge.

Need the scoring walkthrough first? See [Phase 2 notebook](phase_02.ipynb).

<div class="alert alert-warning" role="alert">
    <b>⚙️ Prerequisite — run before this notebook</b>
    <br><br>
    <b>1.</b> From the repo root: <code>bash scripts/setup_data.sh</code> &nbsp;(downloads ontology files into <code>data/ontologies/</code>)
    <br>
    <b>2.</b> Start the server with <code>make notebook</code> &nbsp;(keeps the working directory at the repo root)
</div>

In [1]:
import os
from pathlib import Path

import rich
from rich import inspect
from rich import pretty; pretty.install()

import zlabel
from zlabel.ground import expression_lookup, grounds_under

print(f"zlabel {zlabel.__version__}")

# Resolve data/ontologies relative to the repo root (the server starts there via `make notebook`).
PACKAGE = "zlabel"
ROOT_DIR = Path(os.getcwd().split(PACKAGE, 1)[0]) / PACKAGE
DATA_DIR = ROOT_DIR / "data/ontologies"
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Data not found at {DATA_DIR.resolve()}.  Run: bash scripts/setup_data.sh"
    )

# Labeler bundles the three data authorities + panels.yaml behind one constructor. We also
# load ZFA + ZFIN expression standalone so section 1 can show the grounding primitives directly.
zfa_ontology = zlabel.load_zfa(DATA_DIR / "zfa.obo")
expression_map = zlabel.load_zfin_expression(DATA_DIR / "zfin_wildtype_expression.txt")
panels = zlabel.load_panels(Path(zlabel.__file__).parent / "panels.yaml")
muscle_anchor = next(p for p in panels if p.bucket == "muscle").ontology_anchor

lab = zlabel.Labeler(stage_hpf=48)  # 48 hpf = Long-pec, the keystone sample stage
print(f"ZFA terms: {zfa_ontology.number_of_nodes():,}  ·  genes with expression: {len(expression_map):,}")
print(f"Labeler ready (stage_hpf=48).\nmuscle anchor = {set(muscle_anchor)}")

zlabel 0.1.0


ZFA terms: 3,161  ·  genes with expression: 14,485
Labeler ready (stage_hpf=48).
muscle anchor = {'ZFA:0000548'}


## 1. Grounding — does a marker express where the bucket says it should?

A panel score says *which* bucket a cluster's markers belong to. **Grounding** checks that claim
against reality: do those markers actually express in the right anatomy in a live zebrafish?

Two pure functions do the work:

- **`expression_lookup(expr, symbol)`** — fetch a gene's curated ZFIN wildtype-expression records
  (each one a ZFA anatomy term + a developmental-stage range).
- **`grounds_under(zfa, zfa_id, anchor)`** — walk the ZFA `is_a` / `part_of` graph to test whether
  an expression site sits at or under the bucket's anchor term.

Grounding is **anchor-relative**: the same marker grounds under its own tissue and not under others.

In [2]:
rich.inspect(grounds_under, help=True, docs=True, methods=True, private=True)

╭──────────────────────────────── <function grounds_under at 0x7f79861a93a0> ────────────────────────────────╮
│ def grounds_under(zfa_ontology: 'nx.MultiDiGraph', rec_zfa_id: 'str', anchor: 'frozenset[str]') -> 'bool': │
│                                                                                                            │
│ Whether a ZFA anatomy term sits at or under a set of anchor terms.                                         │
│                                                                                                            │
│ Checks three cases in order: the term is in the anchor (self-match);                                       │
│ the term is not in the loaded ontology (absent -> False, not a crash);                                     │
│ the term's is_a+part_of ancestors include any anchor member.                                               │
│                                                                                                            │
│ This is a per-record predicate. label.py counts how many of the winner's                                   │
│ matched markers ground under the anchor (aggregating across records).                                      │
│                                                                                                            │
│ Args:                                                                                                      │
│     zfa_ontology (nx.MultiDiGraph): ZFA ontology graph from data.load_zfa.                                 │
│     rec_zfa_id (str): The ZFA id from a ZFIN expression record.                                            │
│     anchor (frozenset[str]): The bucket's ontology anchor ids. May be empty                                │
│         for state panels; grounds_under always returns False for an empty                                  │
│         anchor.                                                                                            │
│                                                                                                            │
│ Returns:                                                                                                   │
│     bool: True when rec_zfa_id is in anchor or is a descendant of any                                      │
│     anchor member under is_a+part_of edges.                                                                │
│                                                                                                            │
│ 38 attribute(s) not shown. Run inspect(inspect) for options.                                               │
╰────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
rich.inspect(expression_lookup, help=True, docs=True, methods=True, private=True)

╭──────────────────────────────── <function expression_lookup at 0x7f79861a9300> ─────────────────────────────────╮
│ def expression_lookup(expression_map: 'Mapping[str, list[ZfinExpressionRecord]]', symbol: 'str') ->             │
│ 'list[ZfinExpressionRecord]':                                                                                   │
│                                                                                                                 │
│ Return all ZFIN wildtype-expression records for a gene symbol.                                                  │
│                                                                                                                 │
│ Lowercases the symbol before lookup to match the expression_map key convention                                  │
│ (data.load_zfin_expression lowercases keys at load time). Returns an empty                                      │
│ list when the gene has no curated expression data — not an error.                                               │
│                                                                                                                 │
│ Args:                                                                                                           │
│     expression_map (Mapping[str, list[ZfinExpressionRecord]]): From                                             │
│         data.load_zfin_expression; maps lowercased gene symbol to records.                                      │
│     symbol (str): The resolved current ZFIN symbol to look up.                                                  │
│                                                                                                                 │
│ Returns:                                                                                                        │
│     list[ZfinExpressionRecord]: All expression records for the gene, or                                         │
│     an empty list when the gene is absent from the map.                                                         │
│                                                                                                                 │
│ 38 attribute(s) not shown. Run inspect(inspect) for options.                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
zfa_ontology.nodes.get("ZFA:0009258")


{
    'name': 'angioblastic mesenchymal cell',
    'namespace': 'zebrafish_anatomy',
    'subset': ['cell_slim'],
    'synonym': ['"angioblast" EXACT []'],
    'xref': ['CL:0000566', 'TAO:0009258'],
    'is_a': ['ZFA:0009020'],
    'relationship': ['develops_from ZFA:0009081', 'end ZFS:0000044', 'start ZFS:0000000']
}

**Why isn't expr the data backbone for a class and it has a .get (expression_lookup) or something? Feels like we're missing an opportunity here and being overly functional... ?**

In [5]:
expression_map["kdrl"][0]


ZfinExpressionRecord(
    zfa_id='ZFA:0009258',
    zfa_name='angioblastic mesenchymal cell',
    start_stage='Segmentation:10-13 somites',
    end_stage='Segmentation:10-13 somites'
)

In [6]:
# myod1 is a muscle master regulator. Its in-vivo expression records land in muscle anatomy.
records = expression_lookup(expression_map, "myod1")
print(f"myod1 has {len(records):,} ZFIN expression records.  A few of the anatomy terms:")
seen = []
for record in records:
    if record.zfa_id not in seen:
        seen.append(record.zfa_id)
        print(f"   {record.zfa_id}  {zlabel.term_name(zfa_ontology, record.zfa_id)}")
    if len(seen) == 10:
        break

# Anchor-relative: myod1 grounds under muscle, not under blood.
blood_anchor = next(p for p in panels if p.bucket == "blood_erythroid").ontology_anchor
under_muscle = any(grounds_under(zfa_ontology, record.zfa_id, muscle_anchor) for record in records)
under_blood = any(grounds_under(zfa_ontology, record.zfa_id, blood_anchor) for record in records)
print(f"\nmyod1 grounds under muscle anchor {set(muscle_anchor)}: {under_muscle}")
print(f"myod1 grounds under blood anchor  {set(blood_anchor)}: {under_blood}")

myod1 has 960 ZFIN expression records.  A few of the anatomy terms:
   ZFA:0000003  adaxial cell
   ZFA:0001058  caudal fin
   ZFA:0009117  fast muscle cell
   ZFA:0001114  head
   ZFA:0000041  mesoderm
   ZFA:0000135  notochord
   ZFA:0000255  paraxial mesoderm
   ZFA:0001161  pectoral fin
   ZFA:0001307  pharyngeal musculature
   ZFA:0001342  presumptive intervening zone

myod1 grounds under muscle anchor {'ZFA:0000548'}: True
myod1 grounds under blood anchor  {'ZFA:0005023', 'ZFA:0000007'}: False


## 2. The decision — the whole loop in one call

`Labeler.label()` normalizes the markers, scores them, grounds the winner, checks the stage, and
emits a `Label` evidence packet. Here is the keystone 48 hpf muscle cluster (note the input uses the
old symbol `mylz2`, which normalizes to `mylpfa` before scoring).

In [7]:
rich.inspect(lab.label, help=True, docs=True, methods=True, private=True)

╭─ <bound method Labeler.label of <zlabel.label.Labeler object at 0x7f7985365160>> ─╮
│ def Labeler.label(markers: 'list[str]') -> 'Label':                               │
│                                                                                   │
│ Label one cluster from its marker gene list.                                      │
│                                                                                   │
│ Normalises the markers via the loaded synonym map, scores them against            │
│ all panels, then calls decide() with the loaded ZFA ontology and ZFIN             │
│ expression data.                                                                  │
│                                                                                   │
│ Args:                                                                             │
│     markers (list[str]): Marker gene symbols ordered by significance              │
│         (rank 1 = most significant = index 0). May use old ZFIN names;            │
│         they are normalised via the GAF synonym map before scoring.               │
│                                                                                   │
│ Returns:                                                                          │
│     Label: Evidence packet with bucket, confidence, grounding evidence,           │
│     and next_step.                                                                │
│                                                                                   │
│ 28 attribute(s) not shown. Run inspect(inspect) for options.                      │
╰───────────────────────────────────────────────────────────────────────────────────╯

In [8]:
KEYSTONE = ["mylz2", "acta1b", "tnnt3a", "myod1", "myog", "kdrl"]  # a 48 hpf fast-muscle cluster
normalized_markers = zlabel.label.normalize_markers(KEYSTONE, lab._synonyms)
symbols = [next(iter(normalized_marker.symbols)) for normalized_marker in normalized_markers if normalized_marker.status == zlabel.label.STATUS_RESOLVED]
scores = zlabel.label.score_markers(normalized_markers, lab._panels)

In [9]:
scores[0], scores[1], scores[2]


(
    BucketScore(
        bucket='muscle',
        score=0.8922108454754891,
        germ_layer='mesoderm',
        tissue='muscle',
        lineage='skeletal muscle',
        kind='identity',
        matched_markers=(
            MatchedMarker(input='mylz2', symbol='mylpfa', rank=1, weight=1.0),
            MatchedMarker(input='acta1b', symbol='acta1b', rank=2, weight=0.6309297535714575),
            MatchedMarker(input='tnnt3a', symbol='tnnt3a', rank=3, weight=0.5),
            MatchedMarker(input='myod1', symbol='myod1', rank=4, weight=0.43067655807339306),
            MatchedMarker(input='myog', symbol='myog', rank=5, weight=0.38685280723454163)
        ),
        total_weight=3.304666305987414
    ),
    BucketScore(
        bucket='endothelium',
        score=0.10778915452451095,
        germ_layer='mesoderm',
        tissue='vasculature',
        lineage='endothelium',
        kind='identity',
        matched_markers=(
            MatchedMarker(input='kdrl', symbol='kdrl', ran

In [10]:
zlabel.label.decide(
            scores,
            anchors=lab._anchors,
            expression_map=lab._expression_map,
            zfa_ontology=lab._zfa_ontology,
            stage_hpf=lab.stage_hpf,
            symbols=symbols,
            information_content=lab._information_content,
        )


Label(
    bucket='posterior hypaxial muscle',
    levels=(
        'musculature system',
        'portion of tissue',
        'trunk musculature',
        'muscle',
        'hypaxialis',
        'posterior hypaxial muscle'
    ),
    depth=6,
    abstained=False,
    confidence='high',
    confidence_score=0.92,
    confidence_components={'coherence': 1.0, 'margin': 1.0, 'grounding': 0.6, 'stage': 1.0},
    ambiguity_flag='none',
    states=(),
    panel_bucket='muscle',
    panel_germ_layer='mesoderm',
    zfa_id='ZFA:0005926',
    panel_scores={
        'muscle': 0.8922108454754891,
        'endothelium': 0.10778915452451095,
        'blood_erythroid': 0.0,
        'cartilage': 0.0,
        'cycling': 0.0,
        'endoderm_gut': 0.0,
        'epidermis': 0.0,
        'germline': 0.0,
        'immune_myeloid': 0.0,
        'mesenchyme': 0.0,
        'neural': 0.0,
        'notochord': 0.0,
        'pigment': 0.0,
        'stress_response': 0.0
    },
    positive_markers=('mylpfa',

In [11]:
KEYSTONE = ["mylz2", "acta1b", "tnnt3a", "myod1", "myog", "kdrl"]  # a 48 hpf fast-muscle cluster
label = lab.label(KEYSTONE)

levels_str = " -> ".join(label.levels) if label.levels else "()"
print(f"bucket           : {label.bucket}")           # named ZFA term, not the panel bucket
print(f"depth            : {label.depth}")            # len(levels); evidence-derived
print(f"levels           : {levels_str}")
print(f"panel_bucket     : {label.panel_bucket}")     # coarse prior that anchored the vote
print(f"panel_germ_layer : {label.panel_germ_layer}")
print(f"convergent_genes : {label.convergent_genes}")  # markers that voted for the named ZFA term
print(f"confidence       : {label.confidence}  (score {label.confidence_score:.3f})")
print(f"zfa_id           : {label.zfa_id}")
print(f"next_step        : {label.next_step}")
print(f"rationale        : {label.rationale}")
print("\nExpression evidence — each supporting marker's grounded anatomy:")
for hit in label.expression_evidence:
    print(f"   {hit.symbol:<8} -> {hit.zfa_id}  {hit.zfa_name}")


bucket           : posterior hypaxial muscle
depth            : 6
levels           : musculature system -> portion of tissue -> trunk musculature -> muscle -> hypaxialis -> posterior hypaxial muscle
panel_bucket     : muscle
panel_germ_layer : mesoderm
convergent_genes : ('mylpfa', 'myod1', 'myog')
confidence       : high  (score 0.920)
zfa_id           : ZFA:0005926
next_step        : subcluster
rationale        : posterior hypaxial muscle (IC 11.82) -- 3/6 markers converge; panel prior: muscle

Expression evidence — each supporting marker's grounded anatomy:
   mylpfa   -> ZFA:0005926  posterior hypaxial muscle
   myod1    -> ZFA:0005926  posterior hypaxial muscle
   myog     -> ZFA:0005926  posterior hypaxial muscle


## 3. The confidence rubric

Confidence is a weighted 0–1 score over four components, each in `[0, 1]`:

- **coherence** (0.40) — rank-weighted strength of the winner’s matched markers
- **margin** (0.30) — how far the winner leads the runner-up bucket
- **grounding** (0.20) — fraction of the winning panel’s matched markers whose ZFIN
  expression grounds under the named ZFA term (or the panel anchor when the vote fails)
- **stage** (0.10) — fraction of those markers on-stage for the sample

For the keystone cluster every component is 1.0 — strong panel, dominant lead, all convergent
markers ground to muscle cell anatomy, all on-stage at Long-pec — so it earns a clean `high`.

**How the four components combine.** The final confidence is a weighted sum of the four component scores, each clipped to `[0, 1]`:

`confidence = 0.40 * coherence + 0.30 * margin + 0.20 * grounding + 0.10 * stage`

Because the weights sum to 1.0, the result is always a clean `0–1` number, which is then read off a fixed ladder: `>= 0.80` is **high**, `>= 0.60` is **medium**, and anything lower is **low**.

**Example A — a clean, convergent call.** Imagine a cluster whose markers all point the same way:

| component | weight | value | contribution |
|-----------|:------:|:-----:|:------------:|
| coherence | 0.40 | 1.00 | 0.40 |
| margin | 0.30 | 0.90 | 0.27 |
| grounding | 0.20 | 1.00 | 0.20 |
| stage | 0.10 | 1.00 | 0.10 |

Summing the contributions gives `0.40 + 0.27 + 0.20 + 0.10 = 0.97`, which clears the 0.80 line, so the call is **high**. Strong markers, a dominant lead over the runner-up, and in-vivo evidence that grounds to the right anatomy on-stage — this is the keystone-cluster shape, and the call is trustworthy.

**Example B — real signal, but hedged.** Now imagine a noisier cluster where the winning bucket is still correct, but its lead is thin and only some markers ground in vivo:

| component | weight | value | contribution |
|-----------|:------:|:-----:|:------------:|
| coherence | 0.40 | 0.70 | 0.28 |
| margin | 0.30 | 0.40 | 0.12 |
| grounding | 0.20 | 0.50 | 0.10 |
| stage | 0.10 | 1.00 | 0.10 |

Here the sum is `0.28 + 0.12 + 0.10 + 0.10 = 0.60`, which lands in the **medium** band. The bucket is still assigned, but the thin margin and partial grounding pull it out of `high`, signalling a human reviewer to look twice.

Two guardrails override this raw arithmetic: a germ-layer rollup (`underclustered`) never reports above **medium**, and a `high` always needs real grounding/stage corroboration — if the in-vivo anatomy contradicts the call, the tier is capped no matter how the weighted sum lands. The weights themselves are provisional; Phase 4 evaluation calibrates them against held-out labels.

In [12]:
WEIGHTS = {"coherence": 0.40, "margin": 0.30, "grounding": 0.20, "stage": 0.10}

print(f"{'component':<11} {'weight':>6}  {'value':>5}  contribution")
print("-" * 48)
for name, w in WEIGHTS.items():
    v = label.confidence_components[name]
    bar = "#" * round(v * 20)
    print(f"{name:<11} {w:>6.2f}  {v:>5.2f}  {bar} {w * v:.2f}")
print("-" * 48)
print(f"{'score':<11} {'':>6}  {'':>5}  -> {label.confidence_score:.2f}  ({label.confidence})")

print("\nTiers:  >= 0.80 high  ·  >= 0.60 medium  ·  else low")
print("Caps (honest by design): a germ-layer rollup never exceeds medium, and high needs real")
print("grounding/stage corroboration — anatomy that contradicts the call blocks high regardless of")
print("stage. The weights are provisional; Phase 4 eval calibrates them.")

component   weight  value  contribution
------------------------------------------------
coherence     0.40   1.00  #################### 0.40
margin        0.30   1.00  #################### 0.30
grounding     0.20   0.60  ############ 0.12
stage         0.10   1.00  #################### 0.10
------------------------------------------------
score                      -> 0.92  (high)

Tiers:  >= 0.80 high  ·  >= 0.60 medium  ·  else low
Caps (honest by design): a germ-layer rollup never exceeds medium, and high needs real
grounding/stage corroboration — anatomy that contradicts the call blocks high regardless of
stage. The weights are provisional; Phase 4 eval calibrates them.


## 4. It never overcalls

The reason to ground at all is to know when *not* to commit. The same `label()` call abstains or
rolls up when the evidence does not converge on one bucket:

- **abstain** (`mixed/unresolved`) — the top contenders span contradictory germ layers.
- **roll up** (`underclustered`) — no single bucket dominates, but the contenders share a germ layer,
  so it backs off to that coarser, honest tier instead of guessing.
- **assign anyway, at lower confidence** — a single weak-but-real marker is still labeled, just not `high`.

In [13]:
cases = {
    "confident (muscle)":      ["mylz2", "acta1b", "tnnt3a", "myod1", "myog"],
    "abstain (mixed)":         ["elavl3", "neurod1", "gata1a", "hbae1.1"],
    "rollup (underclustered)": ["myod1", "myog", "gata1a", "hbae1.1"],
    "weak single signal":      ["elavl3"],
}

print(f"{'case':<26} {'bucket':<17} {'confidence':<11} {'flag':<15} markers")
print("-" * 98)
for name, markers in cases.items():
    r = lab.label(markers)
    print(f"{name:<26} {r.bucket:<17} {(r.confidence or '—'):<11} {r.ambiguity_flag:<15} {', '.join(markers)}")

print("\nneural + blood markers span contradictory germ layers -> honest mixed/unresolved.")
print("muscle + blood share the mesoderm germ layer but neither dominates -> roll up to mesoderm.")
print("a lone neural marker is still assigned (neural) — at medium, not high.")

case                       bucket            confidence  flag            markers
--------------------------------------------------------------------------------------------------
confident (muscle)         posterior hypaxial muscle high        none            mylz2, acta1b, tnnt3a, myod1, myog
abstain (mixed)            mixed/unresolved  —           mixed           elavl3, neurod1, gata1a, hbae1.1
rollup (underclustered)    mesoderm          medium      underclustered  myod1, myog, gata1a, hbae1.1
weak single signal         neural            medium      none            elavl3

neural + blood markers span contradictory germ layers -> honest mixed/unresolved.
muscle + blood share the mesoderm germ layer but neither dominates -> roll up to mesoderm.
a lone neural marker is still assigned (neural) — at medium, not high.


## 5. Identity vs. state (optional)

Cell **identity** (what a cell is) and cell **state** (what it is doing) are orthogonal. A dividing
muscle cell is still muscle, so detected state programs are reported *alongside* the identity call,
never in competition with it.

In [14]:
# Muscle markers + cell-cycle markers (mki67, pcna, top2a hit the cycling state panel).
r = lab.label(["mylz2", "acta1b", "myod1", "mki67", "pcna", "top2a"])
print(f"bucket : {r.bucket}  ({r.confidence})")
print(f"states : {r.states}")
print("\nThe cycling program is reported as a state; the identity call stays muscle.")

bucket : skeletal muscle  (high)
states : ('cycling',)

The cycling program is reported as a state; the identity call stays muscle.


## Phase 3 synthesis — the handoff artifact

The deterministic annotation loop now runs end to end. The handoff artifact is the **`Label` evidence
packet**: a bucket (or honest abstention), a confidence tier with its four-component breakdown, the
grounded in-vivo evidence, and a one-line rationale — everything a human or a future LLM narrator
needs to trust or question the call, with no library internals required.

## What’s next

**Phase 4b — evaluation.** The separate **Phase 4a** added the IC-weighted ZFA convergence namer —
which this notebook's `Labeler` already uses — so panels are now the coarse prior, not the naming
authority. Phase 4b runs `label()` across a labeled atlas (Daniocell’s broad tissues) and reports the
numbers that prove it works: broad-bucket agreement, resolution depth, coverage (non-abstain rate),
and abstention calibration. Phase 4b is also where the provisional confidence weights get calibrated
against ground truth.

Need a refresher on the scored bucket table that feeds this notebook? Revisit
[Phase 2 notebook](phase_02.ipynb).

<div class="alert alert-success" role="alert">
    <b>✅ What you have after Phase 3 (with the Phase 4a namer)</b>
    <br><br>
    The full layer-1 loop: marker genes in, a grounded <code>Label</code> evidence packet out —
    labeled at the deepest ZFA anatomy term the evidence supports, with the coarse panel prior
    kept visible, rolled up or honestly abstained when evidence does not converge.
</div>

In [15]:
# End of notebook.